We want to import data from EIA

Currently, EIA's API contains the following main data sets:

Hourly electricity operating data, including actual and forecast demand, net generation, and the power
flowing between electric systems
408,000 electricity series organized into 29,000 categories
30,000 State Energy Data System series organized into 600 categories
115,052 petroleum series and associated categories
34,790 U.S. crude imports series and associated categories
11,989 natural gas series and associated categories
132,331 coal series and associated categories
3,872 Short-Term Energy Outlook series and associated categories
368,466 Annual Energy Outlook series and associated categories
92,836 International energy series

solar electricity generation from utility-scale plants (PV and CSP) feeding the grid. These are big enough to report as power plants (typically ≥1 MW) and are connected to the transmission/distribution grid. Their output is metered and dispatched by grid operators. Not included: most behind-the-meter (BTM) or rooftop solar on homes and businesses. BTM reduces a customer’s grid demand but usually isn’t counted in this EIA-930 generation series

The APi_Key is obtained on EIA website.

In [ ]:

API="https://api.eia.gov/v2/electricity/rto"; BAS=["PSCO","WACM"]; YEARS=range(2022,2025)

def fetch(route, p):
    out, off = [], 0
    while True:
        q={**p,"length":5000,"offset":off,"sort[0][column]":"period","sort[0][direction]":"asc"}
        j=requests.get(f"{API}/{route}/data/",params=q,timeout=30).json()
        rows=(j.get("response") or {}).get("data") or []
        if not rows: break
        out+=rows
        if len(rows)<5000: break
        off+=5000
    return out

# SOLAR (MWh per hour)
srows=[r for ba in BAS for y in YEARS for r in fetch("fuel-type-data",{
    "api_key":API_KEY,"frequency":"hourly","data[0]":"value",
    "facets[respondent][]":ba,"facets[fueltype][]":"SUN",
    "start":f"{y}-01-01T00","end":f"{y}-12-31T23"})]
solar=(pd.DataFrame(srows)[["period","value"]]
         .assign(value=pd.to_numeric(pd.Series([*map(lambda x: x, pd.DataFrame(srows)["value"])]), errors="coerce").fillna(0.0))
         .groupby("period",as_index=False)["value"].sum()
         .rename(columns={"value":"solar"}))

# DEMAND (MW)
drows=[r for ba in BAS for y in YEARS for r in fetch("region-data",{
    "api_key":API_KEY,"frequency":"hourly","data[0]":"value",
    "facets[respondent][]":ba,"facets[type][]":"D",
    "start":f"{y}-01-01T00","end":f"{y}-12-31T23"})]
demand=(pd.DataFrame(drows)[["period","value"]]
          .assign(value=pd.to_numeric(pd.Series([*map(lambda x: x, pd.DataFrame(drows)["value"])]), errors="coerce").fillna(0.0))
          .groupby("period",as_index=False)["value"].sum()
          .rename(columns={"value":"demand"}))

table=(solar.merge(demand, on="period", how="outer")
            .sort_values("period").reset_index(drop=True))
print(table.head())
print(table.tail())